# 05 · Regression: baseline, linear model, and regularization

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Compare dummy, linear, and ridge regressors using MAE, RMSE, and R².


## Step 1 — Generate continuous labels

Synthetic regression has no privacy restrictions. Split once and keep the test set out of tuning.


In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
X, y = make_regression(n_samples=600, n_features=8, noise=15, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=42)
print(Xtr.shape, Xte.shape)


**Check your result:** Training and test rows are disjoint.


## Step 2 — Compare against a baseline

Ridge adds an L2 penalty. Metrics are meaningful only relative to the target scale and a baseline.


In [ ]:
models = {'mean':DummyRegressor(), 'linear':LinearRegression(),
          'ridge':make_pipeline(StandardScaler(), Ridge(alpha=10.0))}
results = {}
for name, model in models.items():
    pred = model.fit(Xtr, ytr).predict(Xte)
    results[name] = {'mae':mean_absolute_error(yte,pred),
                     'rmse':float(np.sqrt(mean_squared_error(yte,pred))), 'r2':r2_score(yte,pred)}
print(results)
assert results['linear']['mae'] < results['mean']['mae']


**Check your result:** Linear regression beats predicting the mean on this linear dataset.


## Step 3 — Inspect residuals

Residual patterns can reveal bias or missing nonlinear features. Small mean residual alone is insufficient.


In [ ]:
residual = yte - models['linear'].predict(Xte)
print('Residual quantiles:', np.quantile(residual,[0,0.25,0.5,0.75,1]))
assert np.isfinite(residual).all()


**Check your result:** Residuals are finite and vary around zero.


## Practice

Increase the noise, then add correlated features. Compare ridge and linear coefficients and test error.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Regularization can stabilize coefficients with correlated predictors. Choose alpha with training cross-validation; do not promise ridge always improves test error.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
